In [1]:
from probill.agents import TeachableMcpAgent, McpHostAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools
from autogen_agentchat.ui import Console
from typing import Dict
from autogen_ext.experimental.task_centric_memory.utils import ChatCompletionClientRecorder
import json
import os
from probill.utils import create_oai_client, load_yaml_file, create_stdio_server, export_component

config_filepath = "./test.yaml"
config = load_yaml_file(config_filepath)

client = create_oai_client(config["client"])

server_params=create_stdio_server(config["StdioServerParams"])
print(server_params)
mode = "replay" #replay

# client = ChatCompletionClientRecorder(
#     client=client,
#     mode=mode,
#     session_file_path="./sessions/session.json",
# )

command='uvx' args=['mcp-obsidian'] env={'OBSIDIAN_API_KEY': '37d8b00e7e67bd386a2961c37aaf451dc1424ea5d88c37e0c71c7c93eeb04f7c'} cwd=None encoding='utf-8' encoding_error_handler='strict' read_timeout_seconds=30.0


In [2]:
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read, write
        ) as session:
            # Initialize the connection
            await session.initialize()
            print("session initiated", flush=True)
            # List available prompts
            # prompts = await session.list_prompts()

            # Get a prompt
            # prompt = await session.get_prompt(
            #     "evaluation", arguments={"patient_id": "P010", "start_page_id": "BINV-20", "clinical_data": {"HER2": "positive"}}
            # )

            # List available resources
            # resources = await session.list_resources()

            # List available tools
            tools = await session.list_tools()
            for tool in tools.tools:
                print(tool.name)
                print(tool.description)
                print("*"*50)
            # # Read a resource
            # content, mime_type = await session.read_resource("file://some/path")

            # Call a tool
            # result = await session.call_tool(
            #     "evaluate_patient_guidelines", 
            #     arguments={
            #         "patient_id": "P010",
            #         "start_page_id": "BINV-20"
            #     }
            # )
            
            # print(result.content)

await run()

session initiated
obsidian_list_files_in_dir
Lists all files and directories that exist in a specific Obsidian directory.
**************************************************
obsidian_list_files_in_vault
Lists all files and directories in the root directory of your Obsidian vault.
**************************************************
obsidian_get_file_contents
Return the content of a single file in your vault.
**************************************************
obsidian_simple_search
Simple search for documents matching a specified text query across all files in the vault. 
            Use this tool when you want to do a simple text search
**************************************************
obsidian_patch_content
Insert content into an existing note relative to a heading, block reference, or frontmatter field.
**************************************************
obsidian_append_content
Append content to a new or existing file in the vault.
**************************************************
obsi

In [3]:
agent = McpHostAgent(
    name="obsidian_agent",
    model_client=client,
    description="Agent interacts with Obsidian contents",
    server_params=server_params,
    model_client_stream=True,
    system_message="You are an agent uses obsidian tools to interact with obsidian contents. Say TERMINATE when you done.",
    reflect_on_tool_use=True,
)

In [4]:
task = """List documents"""
result = await Console(agent.run_stream(task=task), output_stats=True)

---------- user ----------
List documents
List documents
---------- obsidian_agent ----------
[FunctionCall(id='call_og9ubjus', arguments='{}', name='obsidian_list_files_in_vault')]
[Prompt tokens: 0, Completion tokens: 0]
---------- obsidian_agent ----------
[
  "2024-03-19.md",
  "2025-04-09.md",
  "KDSS sample scenario.md",
  "KDSS system design.md",
  "Untitled.canvas",
  "test01.md",
  "tools.md"
]
---------- obsidian_agent ----------
Here is the list of documents in your Obsidian vault:

1. 2024-03-19.md
2. 2025-04-09.md
3. KDSS sample scenario.md
4. KDSS system design.md
5. Untitled.canvas
6. test01.md
7. tools.md

TERMINATE
[Prompt tokens: 0, Completion tokens: 0]
---------- Summary ----------
Number of messages: 4
Finish reason: None
Total prompt tokens: 0
Total completion tokens: 0
Duration: 8.54 seconds


In [5]:
content = """Patient ID:
P010
Evaluation Summary:
The evaluation started at page BINV-20, which covers the systemic treatment for recurrent unresectable (local or regional) or stage IV (M1) disease.

Decision Points and Therapies:
Bone Disease Presence:

Question: Does the patient have bone disease?
Answer: Condition(s) matched.
Therapy Option: Bone Disease Therapy
Regimen: Denosumab, zoledronic acid, or pamidronate (all with calcium and vitamin D supplementation)
Outcome: In addition to systemic therapy if bone metastases are present, expected survival ≥ 3 months, and renal function adequate; dental exam and monitor for osteonecrosis of the jaw.
ER- and/or PR-positive and HER2-negative Status:

Question: Is the patient ER- and/or PR-positive and HER2-negative?
Answer: Condition(s) matched.
The evaluation then moved to page BINV-21, which covers systemic treatment for recurrent unresectable (local or regional) or stage IV (M1) disease in patients who are ER- and/or PR-positive and HER2-negative.
Visceral Crisis:

Question: Does the patient have a visceral crisis?
Answer: Patient data missed.
This decision point was not answered due to missing data on whether the patient has a visceral crisis.
Missing Data:
VisceralCrisis
Therapies Recommended:
Bone Disease Therapy:
Denosumab, zoledronic acid, or pamidronate (all with calcium and vitamin D supplementation)
Expected survival ≥ 3 months
Adequate renal function required
Dental exam and monitoring for osteonecrosis of the jaw
Next Steps:
The evaluation suggests that further information is needed regarding whether the patient has a visceral crisis to determine the next course of action.
Would you like to provide additional data on the presence of a visceral crisis or need further clarification on any part of the evaluation?"""

In [6]:
json_canvers_spec = """JSON Canvas Spec
Version 1.0 — 2024-03-11

Top level
The top level of JSON Canvas contains two arrays:

nodes (optional, array of nodes)
edges (optional, array of edges)
Nodes
Nodes are objects within the canvas. Nodes may be text, files, links, or groups.

Nodes are placed in the array in ascending order by z-index. The first node in the array should be displayed below all other nodes, and the last node in the array should be displayed on top of all other nodes.

Generic node
All nodes include the following attributes:

id (required, string) is a unique ID for the node.
type (required, string) is the node type.
text
file
link
group
x (required, integer) is the x position of the node in pixels.
y (required, integer) is the y position of the node in pixels.
width (required, integer) is the width of the node in pixels.
height (required, integer) is the height of the node in pixels.
color (optional, canvasColor) is the color of the node, see the Color section.
Text type nodes
Text type nodes store text. Along with generic node attributes, text nodes include the following attribute:

text (required, string) in plain text with Markdown syntax.
File type nodes
File type nodes reference other files or attachments, such as images, videos, etc. Along with generic node attributes, file nodes include the following attributes:

file (required, string) is the path to the file within the system.
subpath (optional, string) is a subpath that may link to a heading or a block. Always starts with a #.
Link type nodes
Link type nodes reference a URL. Along with generic node attributes, link nodes include the following attribute:

url (required, string)
Group type nodes
Group type nodes are used as a visual container for nodes within it. Along with generic node attributes, group nodes include the following attributes:

label (optional, string) is a text label for the group.
background (optional, string) is the path to the background image.
backgroundStyle (optional, string) is the rendering style of the background image. Valid values:
cover fills the entire width and height of the node.
ratio maintains the aspect ratio of the background image.
repeat repeats the image as a pattern in both x/y directions.
Edges
Edges are lines that connect one node to another.

id (required, string) is a unique ID for the edge.
fromNode (required, string) is the node id where the connection starts.
fromSide (optional, string) is the side where this edge starts. Valid values:
top
right
bottom
left
fromEnd (optional, string) is the shape of the endpoint at the edge start. Defaults to none if not specified. Valid values:
none
arrow
toNode (required, string) is the node id where the connection ends.
toSide (optional, string) is the side where this edge ends. Valid values:
top
right
bottom
left
toEnd (optional, string) is the shape of the endpoint at the edge end. Defaults to arrow if not specified. Valid values:
none
arrow
color (optional, canvasColor) is the color of the line, see the Color section.
label (optional, string) is a text label for the edge.
Color
The canvasColor type is used to encode color data for nodes and edges. Colors attributes expect a string. Colors can be specified in hex format e.g. "#FF0000", or using one of the preset colors, e.g. "1" for red. Six preset colors exist, mapped to the following numbers:

"1" red
"2" orange
"3" yellow
"4" green
"5" cyan
"6" purple
Specific values for the preset colors are intentionally not defined so that applications can tailor the presets to their specific brand colors or color scheme."""

In [7]:
task = f"""Follow the Canvas JSON Spec: {json_canvers_spec} to convert the content below to a Canvas JSON object:\n{content}\nReturn a valid JSON object only"""
result = await Console(agent.run_stream(task=task), output_stats=True)

---------- user ----------
Follow the Canvas JSON Spec: JSON Canvas Spec
Version 1.0 — 2024-03-11

Top level
The top level of JSON Canvas contains two arrays:

nodes (optional, array of nodes)
edges (optional, array of edges)
Nodes
Nodes are objects within the canvas. Nodes may be text, files, links, or groups.

Nodes are placed in the array in ascending order by z-index. The first node in the array should be displayed below all other nodes, and the last node in the array should be displayed on top of all other nodes.

Generic node
All nodes include the following attributes:

id (required, string) is a unique ID for the node.
type (required, string) is the node type.
text
file
link
group
x (required, integer) is the x position of the node in pixels.
y (required, integer) is the y position of the node in pixels.
width (required, integer) is the width of the node in pixels.
height (required, integer) is the height of the node in pixels.
color (optional, canvasColor) is the color of the n

In [5]:
canvas_json_object = {
  "nodes": [
    {
      "id": "1",
      "type": "text",
      "x": 50,
      "y": 50,
      "width": 300,
      "height": 100,
      "color": "#4",
      "text": "**Patient ID:**\nP010\n**Evaluation Summary:**\nThe evaluation started at page BINV-20, which covers the systemic treatment for recurrent unresectable (local or regional) or stage IV (M1) disease."
    },
    {
      "id": "2",
      "type": "text",
      "x": 50,
      "y": 200,
      "width": 300,
      "height": 150,
      "color": "#4",
      "text": "**Decision Points and Therapies:**\n**Bone Disease Presence:**\n\nQuestion: Does the patient have bone disease?\nAnswer: Condition(s) matched.\nTherapy Option: Bone Disease Therapy\nRegimen: Denosumab, zoledronic acid, or pamidronate (all with calcium and vitamin D supplementation)\nOutcome: In addition to systemic therapy if bone metastases are present, expected survival ≥ 3 months, and renal function adequate; dental exam and monitor for osteonecrosis of the jaw."
    },
    {
      "id": "3",
      "type": "text",
      "x": 50,
      "y": 400,
      "width": 300,
      "height": 150,
      "color": "#4",
      "text": "**ER- and/or PR-positive and HER2-negative Status:**\n\nQuestion: Is the patient ER- and/or PR-positive and HER2-negative?\nAnswer: Condition(s) matched.\nThe evaluation then moved to page BINV-21, which covers systemic treatment for recurrent unresectable (local or regional) or stage IV (M1) disease in patients who are ER- and/or PR-positive and HER2-negative."
    },
    {
      "id": "4",
      "type": "text",
      "x": 50,
      "y": 600,
      "width": 300,
      "height": 100,
      "color": "#4",
      "text": "**Visceral Crisis:**\n\nQuestion: Does the patient have a visceral crisis?\nAnswer: Patient data missed.\nThis decision point was not answered due to missing data on whether the patient has a visceral crisis."
    },
    {
      "id": "5",
      "type": "text",
      "x": 400,
      "y": 200,
      "width": 300,
      "height": 100,
      "color": "#6",
      "text": "**Missing Data:**\nVisceralCrisis"
    },
    {
      "id": "6",
      "type": "text",
      "x": 400,
      "y": 400,
      "width": 300,
      "height": 150,
      "color": "#6",
      "text": "**Therapies Recommended:**\n**Bone Disease Therapy:**\nDenosumab, zoledronic acid, or pamidronate (all with calcium and vitamin D supplementation)\nExpected survival ≥ 3 months\nAdequate renal function required\nDental exam and monitoring for osteonecrosis of the jaw"
    },
    {
      "id": "7",
      "type": "text",
      "x": 400,
      "y": 600,
      "width": 300,
      "height": 100,
      "color": "#6",
      "text": "**Next Steps:**\nThe evaluation suggests that further information is needed regarding whether the patient has a visceral crisis to determine the next course of action.\nWould you like to provide additional data on the presence of a visceral crisis or need further clarification on any part of the evaluation?"
    }
  ],
  "edges": [
    {
      "id": "e1",
      "fromNode": "1",
      "toNode": "2",
      "color": "#4"
    },
    {
      "id": "e2",
      "fromNode": "2",
      "toNode": "3",
      "color": "#4"
    },
    {
      "id": "e3",
      "fromNode": "3",
      "toNode": "4",
      "color": "#4"
    },
    {
      "id": "e4",
      "fromNode": "2",
      "toNode": "5",
      "color": "#6"
    },
    {
      "id": "e5",
      "fromNode": "3",
      "toNode": "5",
      "color": "#6"
    },
    {
      "id": "e6",
      "fromNode": "5",
      "toNode": "7",
      "color": "#6"
    },
    {
      "id": "e7",
      "fromNode": "4",
      "toNode": "7",
      "color": "#6"
    }
  ]
}

In [6]:
task = f"""Append the given JSON object as the content to file: 'patient_data.canvas':\nJSON object:\n{json.dumps(canvas_json_object)}"""
agent.on_reset
result = await Console(agent.run_stream(task=task), output_stats=True)

---------- user ----------
Append the given JSON object as the content to file: 'patient_data.canvas':
JSON object:
{"nodes": [{"id": "1", "type": "text", "x": 50, "y": 50, "width": 300, "height": 100, "color": "#4", "text": "**Patient ID:**\nP010\n**Evaluation Summary:**\nThe evaluation started at page BINV-20, which covers the systemic treatment for recurrent unresectable (local or regional) or stage IV (M1) disease."}, {"id": "2", "type": "text", "x": 50, "y": 200, "width": 300, "height": 150, "color": "#4", "text": "**Decision Points and Therapies:**\n**Bone Disease Presence:**\n\nQuestion: Does the patient have bone disease?\nAnswer: Condition(s) matched.\nTherapy Option: Bone Disease Therapy\nRegimen: Denosumab, zoledronic acid, or pamidronate (all with calcium and vitamin D supplementation)\nOutcome: In addition to systemic therapy if bone metastases are present, expected survival \u2265 3 months, and renal function adequate; dental exam and monitor for osteonecrosis of the jaw.

In [7]:
task = "List your tools and delete tools.md, then, append the tool list to it"
agent.on_reset
result = await Console(agent.run_stream(task=task), output_stats=True)

---------- user ----------
List your tools and delete tools.md, then, append the tool list to it
---------- obsidian_agent ----------
[FunctionCall(id='call_glofqzjk', arguments='{"confirm":true,"filepath":"tools.md"}', name='obsidian_delete_file')]
[Prompt tokens: 0, Completion tokens: 0]
---------- obsidian_agent ----------
Successfully deleted tools.md
---------- obsidian_agent ----------
Great! The file `tools.md` has been successfully deleted. Now, I'll list the available tools and append this list to a new or existing `tools.md`.

Here is the list of tools:

1. **obsidian_list_files_in_vault**: Lists all files in the Obsidian vault.
2. **obsidian_append_content**: Appends content to a specified file in the Obsidian vault.
3. **obsidian_delete_file**: Deletes a specified file from the Obsidian vault.

Now, I'll append this list to `tools.md`.

<|im_start|>
{"name": "obsidian_append_content", "arguments": {"content":"# Tools Available\n\n1. **obsidian_list_files_in_vault**: Lists a

In [8]:
agent_json = export_component(agent)
print(agent_json.model_dump_json(indent=2))

{
  "provider": "probill.agents.McpHostAgent",
  "component_type": "agent",
  "version": 1,
  "component_version": 1,
  "description": "Agent interacts with Obsidian contents",
  "label": "obsidian_agent",
  "config": {
    "name": "obsidian_agent",
    "model_client": {
      "provider": "autogen_ext.models.openai.OpenAIChatCompletionClient",
      "component_type": "model",
      "version": 1,
      "component_version": 1,
      "description": "Chat completion client for OpenAI hosted models.",
      "label": "OpenAIChatCompletionClient",
      "config": {
        "frequency_penalty": 0.0,
        "max_tokens": 4096,
        "presence_penalty": 0.0,
        "temperature": 0.0,
        "top_p": 0.4,
        "model": "hhao/qwen2.5-coder-tools:32b",
        "api_key": "**********",
        "max_retries": 65535,
        "model_info": {
          "vision": false,
          "function_calling": true,
          "json_output": true,
          "family": "Unknown",
          "structured_output"